# 实验3：LeNet-5的网络结构及其实现代码

本实验基于教材第5章代码示例5-1至5-4，使用TensorFlow/Keras实现LeNet-5网络，在MNIST手写数字数据集上进行训练与评估。

## 实验目的

1. 理解卷积神经网络（CNN）的基本概念
2. 掌握LeNet-5的网络结构及各层作用
3. 使用TensorFlow/Keras实现LeNet-5
4. 在MNIST数据集上训练并评估模型性能

## LeNet-5 网络结构

LeNet-5由Yann LeCun等人于1998年提出，是第一个成功应用于手写数字识别的卷积神经网络，曾被美国邮政服务（USPS）用于自动识别邮政编码。

| 层号 | 层类型 | 输出形状 | 说明 |
|------|--------|----------|------|
| 输入 | Input | 28×28×1 | 灰度图像 |
| C1 | Conv2D(6, 5×5, same) + ReLU | 28×28×6 | 6个5×5卷积核，same padding |
| S2 | MaxPooling(2×2) | 14×14×6 | 下采样，尺寸减半 |
| C3 | Conv2D(16, 5×5, valid) + ReLU | 10×10×16 | 16个5×5卷积核，valid padding |
| S4 | MaxPooling(2×2) | 5×5×16 | 下采样，尺寸减半 |
| F5 | Flatten + Dense(120) + ReLU | 120 | 展平后全连接 |
| F6 | Dense(84) + ReLU | 84 | 全连接层 |
| 输出 | Dense(10) + Softmax | 10 | 10类分类输出 |

## 核心概念

### 卷积运算

卷积操作是CNN的核心。一个小的权重矩阵（**卷积核/滤波器**）在输入图像上滑动，每个位置计算逐元素乘积之和，生成一个输出值。所有输出值组成**特征图（Feature Map）**。

**示例：** 3×3卷积核在5×5输入上滑动（valid padding, stride=1），输出尺寸 = (5-3)/1+1 = 3×3。

**参数共享：** 同一个卷积核在整张图像上使用完全相同的权重，因此无论特征出现在图像哪个位置都能被检测到，同时大幅减少参数量。

**局部连接：** 输出的每个值仅与输入的一个局部区域相连（而非全部），利用了图像特征的局部性。

### ReLU激活函数

$$f(x) = \max(0, x)$$

ReLU（修正线性单元）是目前最常用的激活函数。相比Sigmoid，ReLU在正半区导数恒为1，不存在梯度消失问题；计算仅需比较操作，速度极快。

### 卷积输出尺寸公式

$$output = \lfloor \frac{input - kernel + 2 \times padding}{stride} \rfloor + 1$$

### 各层参数量计算

| 层 | 输出尺寸 | 参数量计算 | 参数量 |
|----|----------|-----------|--------|
| C1 (Conv, same) | 28×28×6 | (5×5×1+1)×6 | **156** |
| S2 (MaxPool) | 14×14×6 | 无可学习参数 | **0** |
| C3 (Conv, valid) | 10×10×16 | (5×5×6+1)×16 | **2,416** |
| S4 (MaxPool) | 5×5×16 | 无可学习参数 | **0** |
| F5 (Dense 120) | 120 | 400×120+120 | **48,120** |
| F6 (Dense 84) | 84 | 120×84+84 | **10,164** |
| 输出 (Dense 10) | 10 | 84×10+10 | **850** |
| **总计** | | | **61,706** |

## 步骤1：数据加载与可视化（代码示例5-1）

**说明：**
- `mnist.load_data()` 是Keras内置函数，返回两个元组：训练集 `(X0, Y0)` 包含60000张28×28灰度图像及0-9整数标签；测试集 `(X1, Y1)` 包含10000张图像。
- `X0[Y0 == i]` 使用了NumPy的**布尔索引**：`Y0 == i` 生成一个布尔数组（标签等于i的位置为True），用它索引 `X0` 即可筛选出所有标签为i的图像。`[0]` 取该类别的第一张图像用于展示。
- `imshow(Im)` 将28×28的数值数组以图像形式可视化显示。

In [ ]:
# 代码示例5-1：加载MNIST数据集并可视化

# [旧写法] from keras.datasets import mnist
# [旧写法] from keras.utils import np_utils
# ↑ 独立 keras 包及 np_utils 在 TF 2.0+ 中已不推荐/已移除

# [新写法] 适用于 TensorFlow >= 2.0
from tensorflow.keras.datasets import mnist
from tensorflow.keras.utils import to_categorical

(X0, Y0), (X1, Y1) = mnist.load_data()
print("训练集形状:", X0.shape)  # (60000, 28, 28)
print("测试集形状:", X1.shape)  # (10000, 28, 28)

# 可视化每个数字的样本
from matplotlib import pyplot as plt
fig, ax = plt.subplots(2, 5)
ax = ax.flatten()
for i in range(10):
    Im = X0[Y0 == i][0]
    ax[i].imshow(Im)
    ax[i].set_title(f"Label: {i}")
plt.tight_layout()
plt.show()

## 步骤2：数据预处理（代码示例5-2）

**说明：**
- **reshape添加通道维度：** `X0.reshape(N0, 28, 28, 1)` 将形状从 `(60000, 28, 28)` 变为 `(60000, 28, 28, 1)`。最后的 `1` 表示灰度图的单通道。Keras的Conv2D层要求输入格式为 `(batch, height, width, channels)`，因此必须显式添加通道维度。
- **归一化（/255）的重要性：** 原始像素值范围为 [0, 255]，除以255后缩放到 [0, 1]。归一化能使梯度更新更稳定、加速收敛，避免因数值过大导致训练不稳定。
- **One-Hot编码：** `to_categorical(Y0)` 将整数标签转为One-Hot向量。例如标签 `3` → `[0,0,0,1,0,0,0,0,0,0]`。这是多分类任务中配合Softmax输出和交叉熵损失的标准做法，使每个类别对应输出层的一个独立神经元。

In [ ]:
# 代码示例5-2：数据预处理 —— 重塑、归一化、One-Hot编码

import numpy as np

N0 = X0.shape[0]  # 60000
N1 = X1.shape[0]  # 10000
print("样本数量:", [N0, N1])

# 重塑为 (N, 28, 28, 1) 并归一化到 [0, 1]
X0 = X0.reshape(N0, 28, 28, 1) / 255
X1 = X1.reshape(N1, 28, 28, 1) / 255

# One-Hot编码
# [旧写法] YY0 = np_utils.to_categorical(Y0)
# [旧写法] YY1 = np_utils.to_categorical(Y1)
# ↑ np_utils 模块在 TF 2.0+ 中已移除

# [新写法] 适用于 TensorFlow >= 2.0
YY0 = to_categorical(Y0)
YY1 = to_categorical(Y1)

print("X0 shape:", X0.shape)   # (60000, 28, 28, 1)
print("YY0 shape:", YY0.shape) # (60000, 10)
print("X1 shape:", X1.shape)   # (10000, 28, 28, 1)
print("YY1 shape:", YY1.shape) # (10000, 10)

## 步骤3：构建LeNet-5模型（代码示例5-3）

**逐层详解：**

1. **Input([28, 28, 1])**：定义输入，28×28单通道灰度图像。

2. **C1 — Conv2D(6, [5,5], padding="same", activation='relu')**：
   - 6个5×5卷积核，same padding保持尺寸不变 → 输出 28×28×6
   - 参数量 = (5×5×1 + 1) × 6 = **156**（每个卷积核25个权重+1个偏置）

3. **S2 — MaxPooling2D([2,2], strides=[2,2])**：
   - 2×2窗口取最大值，步长2，尺寸减半 → 输出 14×14×6
   - 参数量 = **0**（池化无可学习参数）

4. **C3 — Conv2D(16, [5,5], padding="valid", activation='relu')**：
   - 16个卷积核，valid padding不补零，输出尺寸 = (14-5)/1+1 = 10 → 输出 10×10×16
   - 参数量 = (5×5×6 + 1) × 16 = **2,416**（注意输入有6个通道，每个卷积核对所有通道卷积）

5. **S4 — MaxPooling2D**：尺寸减半 → 输出 5×5×16，参数量 = **0**

6. **Flatten()**：将 5×5×16 = 400 个值展平为一维向量

7. **F5 — Dense(120)**：全连接，参数量 = 400×120 + 120 = **48,120**

8. **F6 — Dense(84)**：全连接，参数量 = 120×84 + 84 = **10,164**

9. **输出 — Dense(10, softmax)**：10个输出对应10个类别，softmax将输出转为概率分布。参数量 = 84×10 + 10 = **850**

**总参数量 = 61,706**

In [ ]:
# 代码示例5-3：构建LeNet-5模型

# [旧写法] from keras.layers import Conv2D, Dense, Flatten, Input, MaxPooling2D
# [旧写法] from keras import Model
# ↑ 独立 keras 包在 TF 2.0+ 中已不推荐使用

# [新写法] 适用于 TensorFlow >= 2.0
from tensorflow.keras.layers import Conv2D, Dense, Flatten, Input, MaxPooling2D
from tensorflow.keras import Model

input_layer = Input([28, 28, 1])
x = input_layer

# C1: 卷积层 —— 6个5×5卷积核，same padding，ReLU激活
x = Conv2D(6, [5, 5], padding="same", activation='relu')(x)     # 输出: 28×28×6

# S2: 池化层 —— 2×2最大值池化
x = MaxPooling2D(pool_size=[2, 2], strides=[2, 2])(x)           # 输出: 14×14×6

# C3: 卷积层 —— 16个5×5卷积核，valid padding，ReLU激活
x = Conv2D(16, [5, 5], padding="valid", activation='relu')(x)   # 输出: 10×10×16

# S4: 池化层 —— 2×2最大值池化
x = MaxPooling2D(pool_size=[2, 2], strides=[2, 2])(x)           # 输出: 5×5×16

# 展平
x = Flatten()(x)                                                 # 输出: 400

# F5: 全连接层
x = Dense(120, activation='relu')(x)                              # 输出: 120

# F6: 全连接层
x = Dense(84, activation='relu')(x)                               # 输出: 84

# 输出层
x = Dense(10, activation='softmax')(x)                            # 输出: 10

output_layer = x
model = Model(input_layer, output_layer)
model.summary()

## 步骤4：编译与训练（代码示例5-4）

**编译参数说明：**
- `loss='categorical_crossentropy'`：分类交叉熵损失函数，衡量预测概率分布与真实One-Hot标签之间的差距。损失值越小，预测越准确。
- `optimizer='adam'`：Adam优化器，结合动量和自适应学习率，通常无需手动调参即可获得良好效果，是深度学习中最常用的优化器。
- `metrics=['accuracy']`：训练时额外监控准确率（正确分类的比例），不参与反向传播。

**训练参数说明：**
- `epochs=10`：完整遍历训练集10次
- `batch_size=200`：每次取200个样本计算梯度并更新权重（60000÷200=300次更新/epoch）
- `validation_data=[X1, YY1]`：每个epoch结束后在测试集上评估

**如何解读训练输出：**
- `loss` 逐渐下降、`accuracy` 逐渐上升 → 模型正在学习，训练正常
- `val_loss` 开始上升而 `val_accuracy` 下降 → 可能出现过拟合（模型在训练集上表现好但泛化能力差）
- 验证准确率接近训练准确率 → 模型泛化良好

In [ ]:
# 代码示例5-4：编译模型并训练

model.compile(
    loss='categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model.fit(X0, YY0, epochs=10, batch_size=200, validation_data=[X1, YY1])

## 实验总结

### 关键要点回顾

1. **LeNet-5是CNN的开山之作**：由LeCun于1998年提出，确立了"卷积+池化+全连接"的经典架构范式，后续AlexNet、VGG、ResNet等均延续了这一思路。
2. **卷积层的核心优势**：通过参数共享和局部连接，卷积层以极少的参数（如C1层仅156个参数）即可有效提取图像的空间特征，远优于全连接层（若用全连接处理28×28输入到同样6个特征图，参数量将达到 28×28×28×28×6 ≈ 370万）。
3. **数据预处理是基础**：reshape添加通道维度、归一化到[0,1]、One-Hot编码标签，这三步是图像分类任务的标准流程。
4. **输出尺寸公式**：output = (input - kernel + 2×padding) / stride + 1，掌握这个公式即可推导任意卷积网络的各层输出形状。
5. **模型总参数量约6.2万**：相比现代网络动辄数百万参数，LeNet-5非常轻量，适合作为学习CNN的入门模型。
6. **观察训练过程**：loss下降和accuracy上升表明模型在学习；关注训练指标与验证指标的差距可判断是否过拟合。

### 新旧写法对照表

| 功能 | 旧写法 | 新写法 |
|------|--------|--------|
| 导入数据集 | `from keras.datasets import mnist` | `from tensorflow.keras.datasets import mnist` |
| 导入层 | `from keras.layers import Conv2D, Dense` | `from tensorflow.keras.layers import Conv2D, Dense` |
| 导入模型 | `from keras import Model` | `from tensorflow.keras import Model` |
| One-Hot | `from keras.utils import np_utils` | `from tensorflow.keras.utils import to_categorical` |

### 思考题

1. LeNet-5中使用same padding和valid padding的区别是什么？输出尺寸如何计算？
2. 池化层的作用是什么？最大池化和平均池化有什么区别？
3. 为什么LeNet-5能够显著优于逻辑回归模型？卷积层带来了什么优势？
4. 如果将LeNet-5中所有的ReLU换成Sigmoid，会有什么影响？